# `PEPit.jl` tutorial: Accelerated proximal point for monotone inclusions

Before we do anything let us load the necessary packages!

In [1]:
using PEPit, OrderedCollections, Mosek, MosekTools, OffsetArrays

Let us start with defining an empty PEP, which we will construct step by step.

In [2]:
problem = PEP()

PEP(AbstractFunction[], Point[], Constraint[], Expression[], PSDMatrix[], nothing)

Consider the monotone inclusion problem

$$ \mathrm{Find}\, x:\, 0\in Ax, $$

where $A$ is maximally monotone. We denote $J_A = (I + A)^{-1}$ the resolvent of $A$. Let us define this operator class in `PEPit` next.

In [3]:
A = declare_function!(problem, MonotoneOperator, OrderedDict())

MonotoneOperator(PEPFunction(1, true, OrderedDict{PEPFunction, Float64}(PEPFunction(#= circular reference @-2 =#) => 1.0), false, Tuple{Point, Point, Expression}[], Tuple{Point, Point, Expression}[], Constraint[], PSDMatrix[], PSDMatrix[], 0))

**Algorithm**: The algorithm that we want to investigate is the accelerated proximal point is described as follows, for $t \in \{ 0, \dots, n-1\}$

$$
\begin{aligned}
    x_{t+1} &= J_{\alpha A}(y_t), \\
    y_{t+1} &= x_{t+1} + \frac{t}{t+2}(x_{t+1} - x_{t}) - \frac{t}{t+2}(x_t - y_{t-1}),
\end{aligned}
$$

where $x_0=y_0=y_{-1}$,  where $\alpha$ is a positive scalar. 

Our goal is to compute the smallest possible $\tau(n, \alpha)$ such that the guarantee

$$ \|x_n - y_n\|^2 \leqslant \tau(n, \alpha) \|x_0 - x_\star\|^2, $$

is valid, where $A x_\star \ni 0$ and $x_0$ is the initial point that we pick.

For our tutorial, let us take $\alpha = 2.0$ and $n=10$.

In [4]:
alpha = 2

n = 10

10

We next define the optimal point $x_\star \equiv$ `xs` that  satisfies $0 \in Ax_\star$. and the initial point $x_0 \equiv $ `x0`,

In [5]:
xs = stationary_point!(A)

x0 = set_initial_point!(problem)

Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)

The next stage is defining the initial condition $\|x_0 - x_\star\|^2$.

In [6]:
set_initial_condition!(problem, (x0 - xs)^2 <= 1)

1-element Vector{Constraint}:
 Constraint(Expression(9, false, OrderedDict{Any, Float64}((Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing), Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)) => 1.0, (Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing), Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing)) => -1.0, (Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing), Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)) => -1.0, (Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing), Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing)) => 1.0, 1 => -1.0), nothing, nothing), "inequality", 0, nothing, nothing)

Now it is time to implement our algorithm in a loop.

In [7]:
x = OffsetVector(fill(x0, n + 1), 0:n)
y = OffsetVector(fill(x0, n + 1), 0:n)

for i in 0:(n - 2)
    x[i + 1], _, _ = proximal_step!(y[i + 1], A, alpha)
    y[i + 2] = x[i + 1] + i / (i + 2) * (x[i + 1] - x[i]) - i / (i + 2) * (x[i] - y[i])
end
x[n], _, _ = proximal_step!(y[n], A, alpha)

(Point(140, false, OrderedDict{Point, Float64}(Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing) => 1.0, Point(10, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 2, nothing) => -0.40000000000000036, Point(24, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 3, nothing) => -0.8000000000000003, Point(38, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 4, nothing) => -1.2000000000000008, Point(52, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 5, nothing) => -1.6000000000000005, Point(66, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 6, nothing) => -1.9999999999999998, Point(80, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 7, nothing) => -2.4, Point(94, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 8, nothing) => -2.80

Not it is time to define our performance metric, which is $\|x_n - y_n\|^2$.

In [8]:
set_performance_metric!(problem, (x[n] - y[n])^2)

1-element Vector{Expression}:
 Expression(143, false, OrderedDict{Any, Float64}((Point(136, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 11, nothing), Point(136, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 11, nothing)) => 4.0), nothing, nothing)

We have everything to build up the PEP. Now time to solve!

In [9]:
τ_PEPit = solve!(problem, solver=Mosek.Optimizer, verbose=true)

 💻 PEPit:  Setting up the problem: size of the main PSD matrix: 12x12
 💻 PEPit:  Setting up the problem: performance measure is minimum of 1 element(s)
 💻 PEPit:  Setting up the problem: Adding initial conditions and general constraints ...
 💻 PEPit:  Setting up the problem: initial conditions and general constraints (1 constraint(s) added)
 💻 PEPit:  Setting up the problem: interpolation conditions for 1 function(s)
		 function 1 : 55 scalar constraint(s) added
 💻 PEPit:  Compiling SDP
 💻 PEPit:  Calling SDP solver
Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 57              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 12              
  Matrix variables       : 1 (scalarized: 78)
  Integer variables      : 0               

Optimizer

0.01000002690469136

If everything is well, we should get `τ_PEPit = 0.01`  for `n = 10` and `alpha = 2`.

What does the theory say? A tight theoretical worst-case guarantee can be found in [1, Theorem 4.1],
for $n \geqslant 1$,

$$ \|x_n - y_{n-1}\|^2 \leqslant  \frac{1}{n^2}  \|x_0 - x_\star\|^2. $$

In [10]:
τ_theory = 1 / (n^2)

@info "💻  PEPit guarantee: τ_PEPit = $(round(τ_PEPit, digits=6))"

@info "📚  Theoretical guarantee: τ_theory = $(round(τ_theory, digits=6))"

[ Info: 💻  PEPit guarantee: τ_PEPit = 0.01
[ Info: 📚  Theoretical guarantee: τ_theory = 0.01


We should see: 

````Julia
PEPit guarantee: τ_PEPit = 0.01

Theoretical guarantee: τ_theory = 0.01
````

So the guarantee that we found via `PEPit` is the same as theoretical guarantee!

**Reference**:

[1] D. Kim (2021). Accelerated proximal point method for maximally monotone operators.
Mathematical Programming, 1-31.
<https://arxiv.org/pdf/1905.05149v4.pdf>